# Hyperparameter tuning XGBOOST (death as a reference)

## 0. Package loading and installation

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1FMHIud9Hi0rIxnMqRfRFdzBQpKEzI796

In [1]:
# Commented out IPython magic to ensure Python compatibility.
# For Jupyter/Colab notebooks
%reset -f
import gc
gc.collect()

import numpy as np
import pandas as pd
import time

#conda install -c conda-forge \
#    numpy \
#    scipy \
#    pandas=2.2 \
#    pyarrow=15 \
#    scikit-survival \
#    spyder

# conda install -c conda-forge fastparquet
# conda install -c conda-forge xgboost
# conda install -c conda-forge pytorch cpuonly
# conda install -c pytorch pytorch cpuonly
# conda install -c conda-forge matplotlib
# conda install -c conda-forge seaborn
# conda install spyder-notebook -c spyder-ide
# conda install notebook nbformat nbconvert
# conda install -c conda-forge xlsxwriter
# conda install -c conda-forge shap

# import subprocess, sys

# subprocess.check_call([
#     sys.executable,
#     "-m",
#     "pip",
#     "install",
#     "matplotlib"
# ])

# subprocess.check_call([
#     sys.executable,
#     "-m",
#     "pip",
#     "install",
#     "seaborn"
# ])

print("numpy:", np.__version__)


from sksurv.metrics import (
    concordance_index_ipcw,
    brier_score,
    integrated_brier_score
)
from sksurv.util import Surv

#Dput
def dput_df(df, digits=6):
    data = {
        "columns": list(df.columns),
        "data": [
            [round(x, digits) if isinstance(x, (float, np.floating)) else x
             for x in row]
            for row in df.to_numpy()
        ]
    }
    print(data)


#Glimpse function
def glimpse(df, max_width=80):
    print(f"Rows: {df.shape[0]} | Columns: {df.shape[1]}")
    for col in df.columns:
        dtype = df[col].dtype
        preview = df[col].astype(str).head(5).tolist()
        preview_str = ", ".join(preview)
        if len(preview_str) > max_width:
            preview_str = preview_str[:max_width] + "..."
        print(f"{col:<30} {str(dtype):<15} {preview_str}")
#Tabyl function
def tabyl(series):
    counts = series.value_counts(dropna=False)
    props = series.value_counts(normalize=True, dropna=False)
    return pd.DataFrame({"value": counts.index,
                         "n": counts.values,
                         "percent": props.values})
#clean_names
import re

def clean_names(df):
    """
    Mimic janitor::clean_names for pandas DataFrames.
    - Lowercase
    - Replace spaces and special chars with underscores
    - Remove non-alphanumeric/underscore
    """
    new_cols = []
    for col in df.columns:
        # lowercase
        col = col.lower()
        # replace spaces and special chars with underscore
        col = re.sub(r"[^\w]+", "_", col)
        # strip leading/trailing underscores
        col = col.strip("_")
        new_cols.append(col)
    df.columns = new_cols
    return df

numpy: 2.0.1


## Load data

In [2]:

from pathlib import Path

BASE_DIR = Path(
    r"G:\My Drive\Alvacast\SISTRAT 2023\data\20241015_out\pred1"
)


import pickle

with open(BASE_DIR / "imputations_list_jan26.pkl", "rb") as f:
    imputations_list_jan26 = pickle.load(f)


imputation_nodum_1 = pd.read_parquet(
    BASE_DIR / "imputation_nondum_1.parquet",
    engine="fastparquet"
)

X_reduced_imp0 = pd.read_parquet(
    BASE_DIR / "X_reduced_imp0.parquet",
    engine="fastparquet"
)

imputation_1 = pd.read_parquet(
    BASE_DIR / "imputation_1.parquet",
    engine="fastparquet"
)

In [3]:
from IPython.display import display, HTML
import io
import sys

def fold_output(title, func):
    buffer = io.StringIO()
    sys.stdout = buffer
    func()
    sys.stdout = sys.__stdout__
    
    html = f"""
    <details>
      <summary>{title}</summary>
      <pre>{buffer.getvalue()}</pre>
    </details>
    """
    display(HTML(html))


fold_output(
    "Show imputation_nodum_1 structure",
    lambda: imputation_nodum_1.info()
)

fold_output(
    "Show imputation_1 structure",
    lambda: imputation_1.info()
)

fold_output(
    "Show X_reduced_imp0 structure",
    lambda: X_reduced_imp0.info()
)

In [4]:
if isinstance(imputations_list_jan26, list) and len(imputations_list_jan26) > 0:
    print("First element type:", type(imputations_list_jan26[0]))
    if isinstance(imputations_list_jan26[0], dict):
        print("First element keys:", imputations_list_jan26[0].keys())
    elif isinstance(imputations_list_jan26[0], (pd.DataFrame, np.ndarray)):
        print("First element shape:", imputations_list_jan26[0].shape)


This code block:

1.  **Imports the `pickle` library**: This library implements binary protocols for serializing and de-serializing a Python object structure.
2.  **Specifies the `file_path`**: It points to the `.pkl` file you selected.
3.  **Opens the file in binary read mode (`'rb'`)**: This is necessary for loading pickle files.
4.  **Loads the object**: `pickle.load(f)` reads the serialized object from the file and reconstructs it in memory.
5.  **Prints confirmation and basic information**: It verifies that the file was loaded and shows the type of the loaded object, and some details about the first element if it's a list containing common data structures.

#### Compare databases (transformed and original)

Inspect and compare the column names of two datasets: the first imputation from imputations_list_jan26 (which likely contains dummy variables) and imputation_nodum_1 (which, as its name suggests, probably doesn't have dummy variables).


In [5]:
# Inspect columns of the first imputation
cols_first_imp = imputations_list_jan26[0].columns.tolist()
print("First imputation columns:", cols_first_imp[:10], "... total:", len(cols_first_imp))

# Inspect columns of imputation_no_dum
cols_nodum = imputation_nodum_1.columns.tolist()
print("No-dum columns:", cols_nodum[:10], "... total:", len(cols_nodum))

# Compare overlap
common_cols = set(cols_first_imp).intersection(cols_nodum)
missing_in_imp = [c for c in cols_nodum if c not in cols_first_imp]
missing_in_nodum = [c for c in cols_first_imp if c not in cols_nodum]

print("Common columns:", len(common_cols))
print("Missing in imputations_list_jan26:", missing_in_imp)

In [6]:
# Inspect columns of the first imputation
cols_first_imp_raw = imputation_1.columns.tolist()
print("First imputation columns:", cols_first_imp_raw[:10], "... total:", len(cols_first_imp_raw))

# Compare overlap
common_cols_raw = set(cols_first_imp_raw).intersection(cols_nodum)
missing_in_imp_raw = [c for c in cols_nodum if c not in cols_first_imp_raw]

print("Common columns:", len(common_cols_raw))
print("Missing in imputations_list_jan26:", missing_in_imp_raw)
print(common_cols_raw)

In [7]:
import pandas as pd

# Example: choose a combination of variables that uniquely identify rows
key_vars = ["adm_age_rec3", "porc_pobr", "dit_m"]

# Take one imputation (first element of the list) and merge with the no-dum dataset
df_imp = imputations_list_jan26[0]
df_nodum = imputation_nodum_1

merged_check = pd.merge(
    df_imp[key_vars],
    df_nodum[key_vars],
    on=key_vars,
    how="inner"
)

print(f"Merged rows: {merged_check.shape[0]}")
print("Preview of merged check:")
print(merged_check.head())

#drop merge
del merged_check

In [8]:
import pandas as pd

# Example: choose a combination of variables that uniquely identify rows
key_vars_raw = ['dit_m',
            'readmit_time_from_adm_m',
            'death_time_from_adm_m',
            'adm_age_rec3']
# Take one imputation (first element of the list) and merge with the no-dum dataset
df_raw = imputation_1

merged_check_raw = pd.merge(
    df_imp[key_vars],
    df_raw[key_vars],
    on=key_vars,
    how="inner"
)

print(f"Merged rows: {merged_check_raw.shape[0]}")
print("Preview of merged check:")
print(merged_check_raw.head())
print(f"{(merged_check_raw.shape[0] / imputation_1.shape[0] * 100):.2f}%")
#drop merge
del merged_check_raw

### Create bins for followup (landmarks)

This code prepares your data for survival analysis. It extracts the time until an event (like readmission or death) and whether that event actually happened for each patient from the df_nodum dataset. Then, it automatically creates a set of important time points, called an 'evaluation grid', which are specific moments to assess the model's performance on both readmission and death outcomes.


In [9]:
import numpy as np

# Required columns for survival outcomes
required = ["readmit_time_from_disch_m", "readmit_event",
            "death_time_from_disch_m", "death_event"]

# Check that df_raw has all required columns
missing = [c for c in required if c not in df_raw.columns]
if missing:
    raise KeyError(f"df_nodum is missing columns: {missing}")

# Create time/event arrays directly from df_raw
time_readm = df_raw["readmit_time_from_disch_m"].to_numpy()
event_readm = (df_raw["readmit_event"].to_numpy() == 1)

time_death = df_raw["death_time_from_disch_m"].to_numpy()
event_death = (df_nodum["death_event"].to_numpy() == 1)

print("Arrays created for df_raw:")
print("Readmission times:", time_readm[:5])
print("Readmission events:", event_readm[:5])
print("Death times:", time_death[:5])
print("Death events:", event_death[:5])

# Build evaluation grids (quantiles of event times)
event_times_readm = time_readm[event_readm]
event_times_death = time_death[event_death]

if len(event_times_readm) < 5 or len(event_times_death) < 5:
    raise ValueError("Too few events in df_raw to build reliable time grids.")

times_eval_readm = np.unique(np.quantile(event_times_readm, np.linspace(0.05, 0.95, 50)))
times_eval_death = np.unique(np.quantile(event_times_death, np.linspace(0.05, 0.95, 50)))

print("Eval times (readmission):", times_eval_readm[:5], "...", times_eval_readm[-5:])
print("Eval times (death):", times_eval_death[:5], "...", times_eval_death[-5:])


 ## Prepare data


First, we eliminated inmortal time bias (dead patients look like without readmission).

This correction is essentially the Cause-Specific Hazard preparation. It is the correct way to handle Aim 3 unless you switch to a Fine-Gray model (which treats death as a specific type of event 2, rather than censoring 0). For RSF/Coxnet, censoring 0 is the correct approach.

In [ ]:
import numpy as np

# Step 1. Extract survival outcomes directly from df_raw
time_readm = df_raw["readmit_time_from_disch_m"].to_numpy()
event_readm = (df_raw["readmit_event"].to_numpy() == 1)

time_death = df_raw["death_time_from_disch_m"].to_numpy()
event_death = (df_raw["death_event"].to_numpy() == 1)

# Step 2. Build structured arrays (Surv objects)
y_surv_readm = np.empty(len(time_readm), dtype=[("event", "?"), ("time", "<f8")])
y_surv_readm["event"] = event_readm
y_surv_readm["time"] = time_readm

y_surv_death = np.empty(len(time_death), dtype=[("event", "?"), ("time", "<f8")])
y_surv_death["event"] = event_death
y_surv_death["time"] = time_death

# Step 3. Replicate across imputations
n_imputations = len(imputations_list_jan26)
y_surv_readm_list = [y_surv_readm for _ in range(n_imputations)]
y_surv_death_list = [y_surv_death for _ in range(n_imputations)]

import numpy as np

def correct_competing_risks(X_list, y_readm_list, y_death_list):
    """
    Adjust survival outcomes for competing risks (death vs. readmission).

    Parameters
    ----------
    X_list : list of pd.DataFrame
        Imputed predictor datasets (same rows across imputations).
    y_readm_list : list of structured arrays
        Surv(event, time) arrays for readmission.
    y_death_list : list of structured arrays
        Surv(event, time) arrays for death.

    Returns
    -------
    y_readm_corrected_list : list of structured arrays
        Corrected readmission outcomes (death treated as censoring).
    """
    corrected = []
    for y_readm, y_death in zip(y_readm_list, y_death_list):
        y_corr = y_readm.copy()
        # If patient died before readmission → censor at death time
        for i in range(len(y_corr)):
            if y_death["event"][i] and y_death["time"][i] < y_corr["time"][i]:
                y_corr["event"][i] = False
                y_corr["time"][i] = y_death["time"][i]
        corrected.append(y_corr)
    return corrected


# Step 4. Apply correction
y_surv_readm_list_corrected = correct_competing_risks(
    imputations_list_jan26,
    y_surv_readm_list,
    y_surv_death_list
)

In [11]:
# Check type and length
type(y_surv_readm_list_corrected), len(y_surv_readm_list_corrected)

# Look at the first element
y_surv_readm_list_corrected[0][:5]   # first 5 rows


array([(False,  68.96774194), ( True,  81.32258065),
       ( True, 116.74193548), ( True,  91.96774194),
       ( True,  31.03225806)], dtype=[('event', '?'), ('time', '<f8')])

In [12]:
glimpse(imputations_list_jan26[0])
print(y_surv_readm_list_corrected[0].shape, y_surv_readm_list_corrected[0].dtype)

## ML

### Advanced Survival Modeling: XGBoost & Stratified Evaluation

In this section, we transition to a Gradient Boosted Decision Tree (GBDT) framework using XGBoost. This approach serves as a robust, non-linear benchmark to validate findings from the neural network, specifically optimized for high-imbalance survival data (approx. 4% death rate).

#### Methodological Framework:
* **Cox-Objective Boosting:** We utilize the `survival:cox` objective, which optimizes the Cox partial log-likelihood within a boosting architecture. This allows the model to learn complex non-linear risk functions and interactions without assuming proportional hazards or requiring manual interaction terms.

* **Stratified 5-Fold Cross-Validation:** To ensure robustness across diverse treatment modalities, we implement `StratifiedKFold` based on Plan Type (Outpatient, Intensive, Residential). This guarantees that every validation fold maintains the same distribution of clinical settings as the full dataset, preventing the model from overfitting to the majority treatment type.

* **Robust Metrics (IPCW & IBS):** Instead of standard AUC, we optimize for **Uno's C-Index (Inverse Probability of Censoring Weighting)**. This metric is statistically consistent for censored data and prevents bias when evaluating long-term outcomes in unbalanced datasets. We additionally calculate the **Integrated Brier Score (IBS)** to assess the calibration of the predicted survival probabilities.

#### Hyperparameter Optimization:
Given the extreme class imbalance, we employ a **Stratified Randomized Search** over a dense parameter grid. This process tunes critical regularization parameters (`min_child_weight`, `gamma`, `reg_alpha`) to prevent overfitting to the majority class (survivors) while maximizing discrimination on the minority class (events).

#### Breslow Estimation:
To bridge the gap between XGBoost's raw risk scores (log-hazards) and interpretable probabilities needed for calibration metrics, we explicitly compute the **Breslow Estimator**. This reconstructs the baseline survival function S0(t), allowing us to project absolute survival probabilities S(t|x) for any patient at any time point.

### Parameter tuning

In [31]:
from IPython.display import display, HTML
import html

def nb_print(*args, sep=" "):
    msg = sep.join(str(a) for a in args)
    display(HTML(f"<pre style='margin:0'>{html.escape(msg)}</pre>"))


In [13]:
#@title ⚡ XGBoost Robust Tuning (Time Tracked & Memory Safe - CPU Only)
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold, ParameterSampler
from sksurv.metrics import concordance_index_ipcw
import time
import gc
import os
from datetime import datetime
import warnings

# Suppress warnings
warnings.filterwarnings("ignore")

# Start Timer
total_start_time = time.time()

# --- CPU CONFIGURATION ---
# Calculate total cores minus 2 (ensuring at least 1 core is used)
N_CORES = max(1, os.cpu_count() - 2)
print(f"Parallel Execution Configured: Using {N_CORES} CPU cores.")

# --- 1. SETUP & DATA ---
print("Preparing data for Robust XGBoost Tuning...")

try:
    if 'imputations_list_jan26' in locals():
        df_tune = imputations_list_jan26[0].copy()
        y_tune_struct = y_surv_death_list[0]
    elif 'imputations_list' in locals():
        df_tune = imputations_list[0].copy()
        y_tune_struct = y_surv_death_list[0]
    else:
        # Fallback
        df_tune = X_train.copy()
        y_tune_struct = y_surv_death_list[0]

    print(f"  Data Shape: {df_tune.shape}")
    print(f"  Target: Death (Events: {y_tune_struct['event'].sum()})")

except Exception as e:
    raise ValueError(f"Data Error: {e}. Please run data loading steps first.")

# --- 2. STRATIFICATION HELPER ---
def get_stratification_labels(df):
    labels = np.zeros(len(df), dtype=int)
    if 'plan_type_corr_pg_pr' in df.columns: labels[df['plan_type_corr_pg_pr'] == 1] = 1
    if 'plan_type_corr_m_pr' in df.columns: labels[df['plan_type_corr_m_pr'] == 1] = 2
    if 'plan_type_corr_pg_pai' in df.columns: labels[df['plan_type_corr_pg_pai'] == 1] = 3
    if 'plan_type_corr_m_pai' in df.columns: labels[df['plan_type_corr_m_pai'] == 1] = 4
    return labels

strat_labels = get_stratification_labels(df_tune)
y_xgb_label = np.where(y_tune_struct['event'], y_tune_struct['time'], -y_tune_struct['time'])

# --- 3. EXHAUSTIVE SEARCH SPACE ---
param_grid = {
    'learning_rate': [0.005, 0.01, 0.02, 0.05, 0.1],
    'max_depth': [3, 4, 5, 6, 8],
    'min_child_weight': [1, 5, 10, 20, 50],
    'subsample': [0.6, 0.7, 0.8, 0.9],
    'colsample_bytree': [0.5, 0.6, 0.7, 0.8],
    'reg_alpha': [0, 0.1, 1, 5, 10],
    'reg_lambda': [0.1, 1, 5, 10, 20],
    'gamma': [0, 0.1, 0.5, 1, 2]
}

N_ITER = 50
param_list = list(ParameterSampler(param_grid, n_iter=N_ITER, random_state=42))

# --- 4. TUNING LOOP ---
print(f"\nStarting Exhaustive Search ({N_ITER} combos)...")
print(f"  Strategy: 5-Fold Stratified CV")
print(f"  Metric: Uno's C-Index (IPCW)")

results = []

for i, params in enumerate(param_list):
    iter_start = time.time()

    # Fixed Parameters & Configuration (Strictly CPU)
    params['objective'] = 'survival:cox'
    params['eval_metric'] = 'cox-nloglik'
    params['tree_method'] = 'hist'
    params['seed'] = 42            # Explicit seed for XGBoost reproducibility
    params['nthread'] = N_CORES    # Explicit CPU threading (Cores - 2)
    params['device'] = 'cpu'       # Hardcoded to CPU, removing GPU overrides
    params['verbosity'] = 0

    # 5-Fold matching your methodology
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=2125)
    fold_scores = []

    for train_idx, val_idx in skf.split(df_tune, strat_labels):
        X_tr, X_va = df_tune.iloc[train_idx], df_tune.iloc[val_idx]
        y_tr_xgb, y_va_xgb = y_xgb_label[train_idx], y_xgb_label[val_idx]
        y_tr_struct, y_va_struct = y_tune_struct[train_idx], y_tune_struct[val_idx]

        dtrain = xgb.DMatrix(X_tr, label=y_tr_xgb)
        dval = xgb.DMatrix(X_va, label=y_va_xgb)

        model = xgb.train(params, dtrain, num_boost_round=1500,
                          evals=[(dval, 'val')], early_stopping_rounds=30,
                          verbose_eval=False)

        risk_scores = model.predict(dval)

        try:
            c_val = concordance_index_ipcw(y_tr_struct, y_va_struct, risk_scores)[0]
            fold_scores.append(c_val)
        except:
            from sksurv.metrics import concordance_index_censored
            c_val = concordance_index_censored(y_va_struct['event'], y_va_struct['time'], risk_scores)[0]
            fold_scores.append(c_val)

        # Clean Memory
        del model, dtrain, dval, risk_scores
        gc.collect()

    # Average & Store
    avg_score = np.mean(fold_scores)
    std_score = np.std(fold_scores)
    results.append({**params, 'Unos_C_Index': avg_score, 'Std_Dev': std_score})

    if (i+1) % 5 == 0:
        elapsed_min = (time.time() - total_start_time) / 60
        best_so_far = max([r['Unos_C_Index'] for r in results])
        print(f"  [{i+1}/{N_ITER}] Best: {best_so_far:.4f} | Current: {avg_score:.4f} | Elapsed: {elapsed_min:.2f} min")

# --- 5. FINALIZE & EXPORT ---
total_duration_min = (time.time() - total_start_time) / 60
print(f"\nTotal Execution Time: {total_duration_min:.2f} minutes")

df_results = pd.DataFrame(results).sort_values(by='Unos_C_Index', ascending=False)
best_config = df_results.iloc[0].to_dict()

timestamp_str = datetime.now().strftime("%Y%m%d_%H%M")
filename = f"_out/XGB_Death_Robust_Tuning_5Fold_{timestamp_str}.csv"

# Ensure directory exists before saving
os.makedirs("_out", exist_ok=True)
df_results.to_csv(filename, index=False)

print(f"\nTuning Complete!")
print(f"  Best C-Index: {best_config['Unos_C_Index']:.4f}")
print(f"Saved to: {filename}")

~11 minutes

In [32]:
nb_print(f"\nTuning Complete!")
nb_print(f"  Best C-Index: {best_config['Unos_C_Index']:.4f}")

In [27]:
import pandas as pd
from IPython.display import HTML, display

# Reset options so Pandas doesn't force everything
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

# Convert DataFrame to HTML and wrap in a scrollable div
html_table = df_results.to_html()
scroll_box = f"""
<div style="max-height:500px; max-width:1000px; overflow-y:auto; overflow-x:auto; border:1px solid #ccc;">
{html_table}
</div>
"""
display(HTML(scroll_box))


,subsample,reg_lambda,reg_alpha,min_child_weight,max_depth,learning_rate,gamma,colsample_bytree,objective,eval_metric,tree_method,seed,nthread,device,verbosity,Unos_C_Index,Std_Dev
16,0.9,0.1,0.1,1,5,0.010,0.1,0.5,survival:cox,cox-nloglik,hist,42,30,cpu,0,0.745007,0.017049
41,0.9,1.0,1.0,10,3,0.010,0.0,0.5,survival:cox,cox-nloglik,hist,42,30,cpu,0,0.744481,0.016503
44,0.7,1.0,0.1,10,4,0.010,0.0,0.6,survival:cox,cox-nloglik,hist,42,30,cpu,0,0.744270,0.015859
32,0.6,1.0,0.0,1,4,0.010,0.1,0.8,survival:cox,cox-nloglik,hist,42,30,cpu,0,0.744245,0.015812
20,0.7,5.0,5.0,10,4,0.005,0.0,0.5,survival:cox,cox-nloglik,hist,42,30,cpu,0,0.744180,0.015198
43,0.8,0.1,0.1,10,3,0.010,0.0,0.8,survival:cox,cox-nloglik,hist,42,30,cpu,0,0.743670,0.015671
12,0.9,1.0,0.1,10,3,0.005,2.0,0.6,survival:cox,cox-nloglik,hist,42,30,cpu,0,0.743611,0.016138
17,0.8,5.0,10.0,1,5,0.010,1.0,0.5,survival:cox,cox-nloglik,hist,42,30,cpu,0,0.743482,0.015742
7,0.8,1.0,10.0,20,8,0.010,2.0,0.5,survival:cox,cox-nloglik,hist,42,30,cpu,0,0.743223,0.016286
19,0.6,0.1,0.1,20,8,0.005,0.0,0.6,survival:cox,cox-nloglik,hist,42,30,cpu,0,0.743158,0.015927


### Optuna

- Multi-objective tuning balances C-index and IBS.
- Uses 5-fold stratified cross-validation.
- Evaluates performance at 5 clinical time horizons.
- Averages time-specific C-indices for robustness.
- Computes survival via Breslow baseline hazard.
- Converts risk scores into survival probabilities.
- Uses IPCW C-index for censoring adjustment.
- Applies early pruning for poor-performing trials.
- Returns Pareto-optimal models, not a single winner.


In [17]:
# @title Optuna Multi-Objective: C-Index (Discrimination) vs IBS (Calibration)
import optuna
import numpy as np
import pandas as pd
import xgboost as xgb
import gc
import os
from sklearn.model_selection import StratifiedKFold
from sksurv.metrics import concordance_index_ipcw, brier_score

# --- 1. SETUP & CPU CONFIGURATION ---
N_CORES = max(1, os.cpu_count() - 2)
N_ITER = 50
EVAL_HORIZONS = [3, 6, 12, 36, 60]

# df_tune & y_tune_struct exist (imputations_list_jan26 already in memory)
df_tune = imputations_list_jan26[0].copy()
y_tune_struct = y_surv_death_list[0]
y_xgb_label = np.where(y_tune_struct['event'], y_tune_struct['time'], -y_tune_struct['time'])

# --- 2. DUAL STRATIFICATION HELPER ---
def get_dual_stratification_labels(df, y_struct):
    labels = np.zeros(len(df), dtype=int)
    if 'plan_type_corr_pg_pr' in df.columns: labels[df['plan_type_corr_pg_pr'] == 1] = 1
    if 'plan_type_corr_m_pr' in df.columns: labels[df['plan_type_corr_m_pr'] == 1] = 2
    if 'plan_type_corr_pg_pai' in df.columns: labels[df['plan_type_corr_pg_pai'] == 1] = 3
    if 'plan_type_corr_m_pai' in df.columns: labels[df['plan_type_corr_m_pai'] == 1] = 4
    
    event_status = y_struct['event'].astype(int)
    return (labels * 10) + event_status

strat_labels_dual = get_dual_stratification_labels(df_tune, y_tune_struct)

# --- 3. HELPER: BRESLOW SURVIVAL ---
def predict_survival_probs_breslow(y_tr, risk_tr, risk_va, eval_times):
    if np.any(risk_tr <= 0):
        risk_tr = np.exp(risk_tr)
        risk_va = np.exp(risk_va)

    order = np.argsort(y_tr['time'])
    t_train = y_tr['time'][order]
    e_train = y_tr['event'][order]
    risk_train_ord = risk_tr[order]
    
    unique_times = np.unique(t_train[e_train])
    baseline_hazard = np.zeros_like(unique_times, dtype=float)
    
    for i, t in enumerate(unique_times):
        at_risk = t_train >= t
        events_at_t = np.sum((t_train == t) & e_train)
        baseline_hazard[i] = events_at_t / np.sum(risk_train_ord[at_risk])
        
    cum_baseline_hazard = np.cumsum(baseline_hazard)
    
    surv_probs = np.zeros((len(risk_va), len(eval_times)))
    for j, tau in enumerate(eval_times):
        valid_idx = np.where(unique_times <= tau)[0]
        H0_t = cum_baseline_hazard[valid_idx[-1]] if len(valid_idx) > 0 else 0.0
        surv_probs[:, j] = np.exp(-H0_t * risk_va) 
        
    return surv_probs

# --- 4. OPTUNA OBJECTIVE ---
def objective(trial):
    params = {
        'objective': 'survival:cox',
        'eval_metric': 'cox-nloglik',
        'tree_method': 'hist',
        'device': 'cpu',
        'nthread': N_CORES,  # <-- Added to parallel processing
        'verbosity': 0,
        'seed': 42, 
        
        'learning_rate': trial.suggest_float('learning_rate', 0.001, 0.02, log=True),
        'max_depth': trial.suggest_int('max_depth', 2, 5),
        'min_child_weight': trial.suggest_int('min_child_weight', 5, 30),
        'subsample': trial.suggest_float('subsample', 0.6, 0.9),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.4, 0.8),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.1, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.1, 15.0, log=True),
        'gamma': trial.suggest_float('gamma', 0.0, 1.0)
    }

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=2125)
    
    fold_c_indices = []
    fold_ib_scores = []
    fold_global_c_indices = [] 

    for fold_idx, (train_idx, val_idx) in enumerate(skf.split(df_tune, strat_labels_dual)):
        X_tr, X_va = df_tune.iloc[train_idx], df_tune.iloc[val_idx]
        y_tr_xgb, y_va_xgb = y_xgb_label[train_idx], y_xgb_label[val_idx]
        y_tr_struct, y_va_struct = y_tune_struct[train_idx], y_tune_struct[val_idx]

        dtrain = xgb.DMatrix(X_tr, label=y_tr_xgb)
        dval = xgb.DMatrix(X_va, label=y_va_xgb)

        model = xgb.train(
            params, dtrain, 
            num_boost_round=1500,
            evals=[(dval, 'val')], 
            early_stopping_rounds=30, 
            verbose_eval=False
        )

        risk_tr = model.predict(dtrain)
        risk_va = model.predict(dval)
        
        # 1. MULTI-HORIZON C-INDEX
        h_c_indices = []
        for tau_val in EVAL_HORIZONS:
            try:
                c_val = concordance_index_ipcw(y_tr_struct, y_va_struct, risk_va, tau=tau_val)[0]
                h_c_indices.append(c_val)
            except:
                pass 
        avg_c_index = np.mean(h_c_indices) if len(h_c_indices) > 0 else 0.5
        
        # 2. GLOBAL C-INDEX 
        try:
            global_c = concordance_index_ipcw(y_tr_struct, y_va_struct, risk_va)[0]
        except:
            global_c = 0.5
        fold_global_c_indices.append(global_c)

        # 3. BRIER SCORE (IBS)
        try:
            surv_probs_va = predict_survival_probs_breslow(y_tr_struct, risk_tr, risk_va, EVAL_HORIZONS)
            _, brier_scores_at_tau = brier_score(y_tr_struct, y_va_struct, surv_probs_va, EVAL_HORIZONS)
            avg_ibs = np.mean(brier_scores_at_tau)
        except:
            avg_ibs = 0.25 

        fold_c_indices.append(avg_c_index)
        fold_ib_scores.append(avg_ibs)
            
        del model, dtrain, dval, risk_tr, risk_va
        gc.collect()

        # Pruning
        current_mean_c = np.mean(fold_c_indices)
        if fold_idx >= 1 and current_mean_c < 0.60:
            raise optuna.TrialPruned()

    trial.set_user_attr("Global_C_Index", np.mean(fold_global_c_indices))
    return np.mean(fold_c_indices), np.mean(fold_ib_scores)


# --- 5. MULTI-OBJECTIVE INITIALIZATION ---
study = optuna.create_study(
    directions=['maximize', 'minimize'], # <-- CORREGIDO PARA QUE SEAN DOS OBJETIVOS
    study_name="XGB_Death_Optuna_Fresh_Search"
)

print(f"\nInitiating search from start, ({N_ITER} combinaciones)...")
study.optimize(objective, n_trials=N_ITER, show_progress_bar=True)

# --- 6. EXTRACTION OF OPTIMAL MODELS ---
print("\nOptimal Models found (Pareto Front):")
best_trials = study.best_trials
for t in best_trials:
    global_c_val = t.user_attrs.get("Global_C_Index", "N/A")
    print(f"Trial {t.number} -> Multi-Horizon C: {t.values[0]:.4f} | IBS: {t.values[1]:.4f} | Global C: {global_c_val:.4f}")

[I 2026-02-23 10:30:53,636] A new study created in memory with name: XGB_Death_Optuna_Fresh_Search
100%|██████████| 50/50 [22:27<00:00, 26.95s/it]


~22 minutes

In [29]:
nb_print("\nOptimal Models found (Pareto Front):")
best_trials = study.best_trials
for t in best_trials:
    global_c_val = t.user_attrs.get("Global_C_Index", "N/A")
    nb_print(f"Trial {t.number} -> Multi-Horizon C: {t.values[0]:.4f} | IBS: {t.values[1]:.4f} | Global C: {global_c_val:.4f}")

In [18]:
import os
import joblib

# Make sure the folder exists before saving
os.makedirs("_input", exist_ok=True)

# --- 7. SAVE THE ENTIRE STUDY TO A .PKL FILE ---
study_filename = f"_input/XGB_Death_Optuna_Study_{timestamp_str}.pkl"

# Save study object
joblib.dump(study, study_filename)
print(f"Study object successfully saved to: {study_filename}")

# (Optional) Force download if running in Google Colab
try:
    from google.colab import files
    files.download(study_filename)
except Exception as e:
    print("Download skipped (not in Colab or browser blocked it).")


In [19]:
# @title Final Model Selection (Euclidean Distance to Ideal Point)
import pandas as pd
import numpy as np
from datetime import datetime

print("Analyzing the Pareto Front...")

# --- Load or reuse study object ---
import os
import joblib
import glob
import re
import datetime as _dt

# If an in-memory study with the specific study_name exists, use it.
# Otherwise, find the most recent saved study file matching the pattern and load it.
if 'study' in globals() and getattr(study, 'study_name', None) == "XGB_Death_Optuna_Fresh_Search":
    # Use the existing in-memory study; no file load required.
    print("Using in-memory study with study_name='XGB_Death_Optuna_Fresh_Search'.")
else:
    # Ensure the input folder exists before searching
    os.makedirs("_input", exist_ok=True)

    # Pattern for saved study files
    pattern = "_input/XGB_Death_Optuna_Study_*.pkl"
    files = glob.glob(pattern)

    if not files:
        raise FileNotFoundError("No saved study files found matching pattern: _input/XGB_Death_Optuna_Study_*.pkl")

    # Extract timestamp from filenames and pick the most recent one
    timestamped_files = []
    for f in files:
        m = re.search(r"XGB_Death_Optuna_Study_(\d{8}_\d{4})\.pkl$", f)
        if m:
            ts = m.group(1)
            try:
                dt = _dt.datetime.strptime(ts, "%Y%m%d_%H%M")
                timestamped_files.append((dt, f))
            except ValueError:
                # Skip files with non-matching timestamp formats
                continue

    if not timestamped_files:
        raise FileNotFoundError("No study files with a valid timestamp found in filenames.")

    # Select the file with the latest timestamp
    latest_file = max(timestamped_files, key=lambda x: x[0])[1]

    # Load the most recent study file
    study_filename = latest_file
    study = joblib.load(study_filename)
    print(f"Loaded study object from most recent file: {study_filename}")

# 1. Extract all optimal models (Pareto Front)
pareto_trials = study.best_trials

# 2. Convert the optimal trials into a DataFrame
pareto_data = []
for t in pareto_trials:
    row = {
        "trial_id": t.number,
        "Multi_Horizon_C_Index": t.values[0],
        "Brier_Score": t.values[1], # Mean time-specific Brier score
        "Global_C_Index": t.user_attrs.get("Global_C_Index", np.nan) # Extracted from user attributes
    }
    # Add the hyperparameters for this specific trial
    row.update(t.params)
    pareto_data.append(row)

df_pareto = pd.DataFrame(pareto_data)

# 3. Calculate the Distance to the Ideal Point (C-Index = 1.0, Brier Score = 0.0)
# The methodological goal is to MINIMIZE this Euclidean distance
df_pareto["Distance_to_Ideal"] = np.sqrt(
    (1.0 - df_pareto["Multi_Horizon_C_Index"])**2 + (df_pareto["Brier_Score"])**2
)

# 4. Sort to find the absolute winner (the "knee point" of the Pareto front)
df_pareto = df_pareto.sort_values("Distance_to_Ideal", ascending=True).reset_index(drop=True)

# --- RESULTS ---
print(f"\nFound {len(df_pareto)} models in the Pareto Front.")

print("\nABSOLUTE WINNER (Optimal Trade-off / Knee Point):")
winner = df_pareto.iloc[0]
print(f"  Trial ID              : {winner['trial_id']}")
print(f"  Multi-Horizon C-Index : {winner['Multi_Horizon_C_Index']:.4f}")
print(f"  Brier Score           : {winner['Brier_Score']:.4f}")
print(f"  Global C-Index        : {winner['Global_C_Index']:.4f}")
print(f"  Distance              : {winner['Distance_to_Ideal']:.4f}")

print("\nWinner Hyperparameters:")
exclude_keys = ["trial_id", "Multi_Horizon_C_Index", "Brier_Score", "Global_C_Index", "Distance_to_Ideal"]
params_winner = {k: v for k, v in winner.items() if k not in exclude_keys}
for k, v in params_winner.items():
    print(f"  {k}: {v}")

# Export the Pareto Front for backup and manuscript reporting
timestamp_str = datetime.now().strftime("%Y%m%d_%H%M")
filename_pareto = f"_out/Pareto_Front_XGB_{timestamp_str}.csv"
os.makedirs("_out", exist_ok=True)
df_pareto.to_csv(filename_pareto, index=False)

print(f"\nPareto Front saved to: {filename_pareto}")

In [20]:
import pandas as pd

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

display(df_pareto.sort_values("Multi_Horizon_C_Index", ascending=False).head(10))

,trial_id,Multi_Horizon_C_Index,Brier_Score,Global_C_Index,learning_rate,max_depth,min_child_weight,subsample,colsample_bytree,reg_alpha,reg_lambda,gamma,Distance_to_Ideal
0,21,0.778628,0.01894,0.74776,0.01521,3,5,0.712648,0.444046,0.905679,0.214096,0.095706,0.222181


In [21]:
from IPython.display import display, HTML

html_content = """
<div style="font-family: Arial; line-height: 1.6;">

<h2>📊 Pareto Front Analysis</h2>

<table style="border-collapse: collapse; width: 100%; font-size: 14px;">
<thead>
<tr style="background-color:#f5f5f5;">
<th style="border:1px solid #ccc; padding:8px;">Component</th>
<th style="border:1px solid #ccc; padding:8px;">Trial 21 (Absolute Winner)</th>
<th style="border:1px solid #ccc; padding:8px;">Interpretation</th>
</tr>
</thead>
<tbody>

<tr>
<td style="border:1px solid #ccc; padding:8px;"><b>Multi-Horizon C-Index</b></td>
<td style="border:1px solid #ccc; padding:8px;">0.7786</td>
<td style="border:1px solid #ccc; padding:8px;">
Trial 21 achieves strong discrimination across clinically relevant horizons (3–60 months), effectively ranking patient risk over time while handling the competing risk nuances.
</td>
</tr>

<tr>
<td style="border:1px solid #ccc; padding:8px;"><b>Brier Score</b></td>
<td style="border:1px solid #ccc; padding:8px;">0.0189</td>
<td style="border:1px solid #ccc; padding:8px;">
Exceptional calibration. The error rate is extremely low, meaning the predicted absolute probabilities are highly reliable and closely match the observed baseline rates.
</td>
</tr>

<tr>
<td style="border:1px solid #ccc; padding:8px;"><b>Global C-Index</b></td>
<td style="border:1px solid #ccc; padding:8px;">0.7478</td>
<td style="border:1px solid #ccc; padding:8px;">
Maintains robust global discrimination over the entire follow-up period, proving the model does not sacrifice overall ranking to achieve its multi-horizon calibration.
</td>
</tr>

<tr>
<td style="border:1px solid #ccc; padding:8px;"><b>Distance to Ideal (C=1, IBS=0)</b></td>
<td style="border:1px solid #ccc; padding:8px;">0.2222</td>
<td style="border:1px solid #ccc; padding:8px;">
This score represents the mathematical "knee point" of the Pareto front, providing an objective, non-arbitrary mathematical basis for selecting this specific trade-off.
</td>
</tr>

</tbody>
</table>

<br>

<h3>🧠 Hyperparameter Robustness Interpretation (Trial 21)</h3>

<table style="border-collapse: collapse; width: 100%; font-size: 14px;">
<thead>
<tr style="background-color:#f5f5f5;">
<th style="border:1px solid #ccc; padding:8px;">Hyperparameter</th>
<th style="border:1px solid #ccc; padding:8px;">Value</th>
<th style="border:1px solid #ccc; padding:8px;">Statistical Meaning</th>
</tr>
</thead>
<tbody>

<tr>
<td style="border:1px solid #ccc; padding:8px;"><b>max_depth</b></td>
<td style="border:1px solid #ccc; padding:8px;">3</td>
<td style="border:1px solid #ccc; padding:8px;">
Consistently shallow trees reduce variance and prevent overfitting, forcing the model to rely on simple, generalizable clinical rules (additive risk) rather than memorizing complex noise.
</td>
</tr>

<tr>
<td style="border:1px solid #ccc; padding:8px;"><b>min_child_weight</b></td>
<td style="border:1px solid #ccc; padding:8px;">5</td>
<td style="border:1px solid #ccc; padding:8px;">
Ensures a sufficient sample size per leaf before making a split, acting as a direct safeguard against the low (~4%) mortality event rate.
</td>
</tr>

<tr>
<td style="border:1px solid #ccc; padding:8px;"><b>gamma</b></td>
<td style="border:1px solid #ccc; padding:8px;">0.096</td>
<td style="border:1px solid #ccc; padding:8px;">
Minimal aggressive pruning. Because the max_depth is already severely constrained (3), the algorithm doesn't need high gamma to prevent overfitting.
</td>
</tr>

<tr>
<td style="border:1px solid #ccc; padding:8px;"><b>learning_rate</b></td>
<td style="border:1px solid #ccc; padding:8px;">0.015</td>
<td style="border:1px solid #ccc; padding:8px;">
A slow, highly stable optimization trajectory that prevents the gradient descent from overshooting the global minimum.
</td>
</tr>

<tr>
<td style="border:1px solid #ccc; padding:8px;"><b>Regularization (α, λ)</b></td>
<td style="border:1px solid #ccc; padding:8px;">α=0.91, λ=0.21</td>
<td style="border:1px solid #ccc; padding:8px;">
L1 penalty (α) dominates over L2. This encourages "sparsity," effectively performing automatic feature selection by dropping uninformative variables, which is optimal for linear-like survival outcomes.
</td>
</tr>

</tbody>
</table>

<p style="margin-top:15px;">
<b>Overall Interpretation:</b> Trial 21 represents a conservative, highly stable configuration optimized for a low-event-rate survival outcome. It balances a strictly shallow architecture with L1-driven feature selection, resulting in an optimal trade-off between predicting time-specific clinical risk (C-Index) and absolute probability accuracy (Brier Score).
</p>

</div>
"""

display(HTML(html_content))

Component,Trial 21 (Absolute Winner),Interpretation
Multi-Horizon C-Index,0.7786,"Trial 21 achieves strong discrimination across clinically relevant horizons (3–60 months), effectively ranking patient risk over time while handling the competing risk nuances."
Brier Score,0.0189,"Exceptional calibration. The error rate is extremely low, meaning the predicted absolute probabilities are highly reliable and closely match the observed baseline rates."
Global C-Index,0.7478,"Maintains robust global discrimination over the entire follow-up period, proving the model does not sacrifice overall ranking to achieve its multi-horizon calibration."
"Distance to Ideal (C=1, IBS=0)",0.2222,"This score represents the mathematical ""knee point"" of the Pareto front, providing an objective, non-arbitrary mathematical basis for selecting this specific trade-off."
Hyperparameter,Value,Statistical Meaning
max_depth,3,"Consistently shallow trees reduce variance and prevent overfitting, forcing the model to rely on simple, generalizable clinical rules (additive risk) rather than memorizing complex noise."
min_child_weight,5,"Ensures a sufficient sample size per leaf before making a split, acting as a direct safeguard against the low (~4%) mortality event rate."
gamma,0.096,"Minimal aggressive pruning. Because the max_depth is already severely constrained (3), the algorithm doesn't need high gamma to prevent overfitting."
learning_rate,0.015,"A slow, highly stable optimization trajectory that prevents the gradient descent from overshooting the global minimum."
"Regularization (α, λ)","α=0.91, λ=0.21","L1 penalty (α) dominates over L2. This encourages ""sparsity,"" effectively performing automatic feature selection by dropping uninformative variables, which is optimal for linear-like survival outcomes."


## Optimism correction

🔟 Take-home messages (what the code does)

- Implements Harrell’s bootstrap optimism correction.
- Uses the final tuned XGBoost Cox model (Trial 21).
- Estimates apparent C-index on full dataset.
- Determines optimal boosting rounds via early stopping.
- Trains final baseline model on 100% of data.
- Runs 100 bootstrap resamples in parallel.
- Retrains model inside each bootstrap sample.
- Computes performance on bootstrap and original data.
- Calculates optimism = apparent_boot − test_original.
- Reports optimism-corrected C-index for internal validation.

🧩 Assumptions (5 key ones)

- Bootstrap samples approximate the data-generating process.
- Model structure and hyperparameters are fixed.
- C-index is appropriate performance metric.
- IPCW assumptions hold for censoring mechanism.
- Sample size is large enough for stable bootstrap estimates.


In [35]:
# @title Harrell's Bootstrap Optimism Correction (Parallelized, CPU-2)
import numpy as np
import pandas as pd
import xgboost as xgb
import gc
import os
from sklearn.utils import resample
from sklearn.model_selection import train_test_split
from sksurv.metrics import concordance_index_ipcw
from joblib import Parallel, delayed
import warnings

warnings.filterwarnings("ignore")

nb_print("Initializing Parallel Harrell's Bootstrap Optimism Correction...")

# --- CPU CONFIGURATION ---
N_CORES = max(1, os.cpu_count() - 2)
nb_print(f"Parallel Execution Configured: Using {N_CORES} CPU cores.")

# --- 1. SET TRIAL 21 HYPERPARAMETERS (ABSOLUTE WINNER) ---
params_winner = {
    'objective': 'survival:cox',
    'eval_metric': 'cox-nloglik',
    'tree_method': 'hist',
    'device': 'cpu',
    'verbosity': 0,
    'seed': 42,
    
    # Trial 21 parameters from Optuna Pareto Front
    'learning_rate': 0.01521,
    'max_depth': 3,
    'min_child_weight': 5,
    'subsample': 0.712648,
    'colsample_bytree': 0.444046,
    'reg_alpha': 0.905679,
    'reg_lambda': 0.214096,
    'gamma': 0.095706
}

B_ITERATIONS = 500 # Standard number of bootstrap iterations for clinical papers

# --- STRATIFICATION HELPER (Safety check to ensure it exists) ---
def get_dual_stratification_labels(df, y_struct):
    labels = np.zeros(len(df), dtype=int)
    if 'plan_type_corr_pg_pr' in df.columns: labels[df['plan_type_corr_pg_pr'] == 1] = 1
    if 'plan_type_corr_m_pr' in df.columns: labels[df['plan_type_corr_m_pr'] == 1] = 2
    if 'plan_type_corr_pg_pai' in df.columns: labels[df['plan_type_corr_pg_pai'] == 1] = 3
    if 'plan_type_corr_m_pai' in df.columns: labels[df['plan_type_corr_m_pai'] == 1] = 4
    event_status = y_struct['event'].astype(int)
    return (labels * 10) + event_status

# Ensure variables are mapped correctly from your environment
strat_labels_dual = get_dual_stratification_labels(df_tune, y_tune_struct)

# --- 2. CALCULATE APPARENT PERFORMANCE ON ORIGINAL DATA ---
nb_print("Calculating apparent performance on the original full dataset...")

# Initial config uses all allocated cores for speed
params_initial = params_winner.copy()
params_initial['nthread'] = N_CORES

X_train_app, X_val_app, y_train_xgb_app, y_val_xgb_app = train_test_split(
    df_tune, y_xgb_label, test_size=0.2, random_state=42, stratify=strat_labels_dual
)

dtrain_app = xgb.DMatrix(X_train_app, label=y_train_xgb_app)
dval_app = xgb.DMatrix(X_val_app, label=y_val_xgb_app)

temp_model = xgb.train(
    params_initial, dtrain_app, 
    num_boost_round=1500, 
    evals=[(dval_app, 'val')], 
    early_stopping_rounds=30, 
    verbose_eval=False
)
optimal_boost_rounds = temp_model.best_iteration

nb_print(f"Optimal boosting rounds determined: {optimal_boost_rounds}")

# Train the definitive baseline model on 100% of the data
dorig = xgb.DMatrix(df_tune, label=y_xgb_label)
baseline_model = xgb.train(
    params_initial, dorig, 
    num_boost_round=optimal_boost_rounds,
    verbose_eval=False
)

risk_orig = baseline_model.predict(dorig)
try:
    c_apparent_orig = concordance_index_ipcw(y_tune_struct, y_tune_struct, risk_orig)[0]
except:
    from sksurv.metrics import concordance_index_censored
    c_apparent_orig = concordance_index_censored(y_tune_struct['event'], y_tune_struct['time'], risk_orig)[0]

nb_print(f"Baseline Apparent C-index: {c_apparent_orig:.4f}")

# --- 3. PARALLEL BOOTSTRAP WORKER FUNCTION ---
def parallel_bootstrap_worker(b, df_original, y_xgb_original, y_struct_orig, params, opt_rounds):
    # CRITICAL: Force 1 thread per XGBoost model to prevent CPU thrashing during multiprocessing
    boot_params = params.copy()
    boot_params['nthread'] = 1 
    
    indices = np.arange(len(df_original))
    boot_indices = resample(indices, replace=True, n_samples=len(indices), random_state=b)
    
    X_boot = df_original.iloc[boot_indices]
    y_xgb_boot = y_xgb_original[boot_indices]
    y_struct_boot = y_struct_orig[boot_indices]
    
    # Recreate DMatrix objects strictly inside the isolated worker
    dboot = xgb.DMatrix(X_boot, label=y_xgb_boot)
    dorig_local = xgb.DMatrix(df_original, label=y_xgb_original)
    
    boot_model = xgb.train(
        boot_params, dboot, 
        num_boost_round=opt_rounds,
        verbose_eval=False
    )
    
    # Evaluate on Bootstrap Sample (Apparent Boot Performance)
    risk_boot = boot_model.predict(dboot)
    try:
        c_boot_app = concordance_index_ipcw(y_struct_boot, y_struct_boot, risk_boot)[0]
    except:
        from sksurv.metrics import concordance_index_censored
        c_boot_app = concordance_index_censored(y_struct_boot['event'], y_struct_boot['time'], risk_boot)[0]
        
    # Evaluate on Original Data (Test Performance)
    risk_test_orig = boot_model.predict(dorig_local)
    try:
        c_boot_test = concordance_index_ipcw(y_struct_boot, y_struct_orig, risk_test_orig)[0]
    except:
        from sksurv.metrics import concordance_index_censored
        c_boot_test = concordance_index_censored(y_struct_orig['event'], y_struct_orig['time'], risk_test_orig)[0]
        
    # Calculate Optimism
    optimism = c_boot_app - c_boot_test
    return optimism

# --- 4. EXECUTE PARALLEL BOOTSTRAP LOOP ---
nb_print(f"\nLaunching {B_ITERATIONS} Parallel Bootstrap Iterations...")

optimism_values = Parallel(n_jobs=N_CORES, verbose=10)(
    delayed(parallel_bootstrap_worker)(
        b, df_tune, y_xgb_label, y_tune_struct, params_winner, optimal_boost_rounds
    ) for b in range(B_ITERATIONS)
)

# --- 5. CALCULATE FINAL CORRECTED METRICS ---
mean_optimism = np.mean(optimism_values)
c_index_corrected = c_apparent_orig - mean_optimism

nb_print("\n--------------------------------------------------")
nb_print("FINAL OPTIMISM-CORRECTED RESULTS")
nb_print("--------------------------------------------------")
nb_print(f"Apparent C-Index (Original Data) : {c_apparent_orig:.4f}")
nb_print(f"Mean Optimism (from {B_ITERATIONS} boots)   : {mean_optimism:.4f}")
nb_print(f"Optimism-Corrected C-Index       : {c_index_corrected:.4f}")
nb_print("--------------------------------------------------")

# Save results to CSV for documentation
os.makedirs("_out", exist_ok=True)
results_df = pd.DataFrame({
    'Metric': ['Apparent_C_Index', 'Mean_Optimism', 'Corrected_C_Index'],
    'Value': [c_apparent_orig, mean_optimism, c_index_corrected]
})
timestamp_str = pd.Timestamp.now().strftime("%Y%m%d_%H%M")
filename = f"_out/XGB_Death_Bootstrap_Optimism_Results_{timestamp_str}.csv"
results_df.to_csv(filename, index=False)
nb_print(f"Results saved successfully to {filename}.")

[Parallel(n_jobs=30)]: Using backend LokyBackend with 30 concurrent workers.
[Parallel(n_jobs=30)]: Done   1 tasks      | elapsed:   31.0s
[Parallel(n_jobs=30)]: Done  12 tasks      | elapsed:   35.4s
[Parallel(n_jobs=30)]: Done  25 tasks      | elapsed:   38.7s
[Parallel(n_jobs=30)]: Done  38 tasks      | elapsed:  1.0min
[Parallel(n_jobs=30)]: Done  53 tasks      | elapsed:  1.2min
[Parallel(n_jobs=30)]: Done  68 tasks      | elapsed:  1.5min
[Parallel(n_jobs=30)]: Done  85 tasks      | elapsed:  1.7min
[Parallel(n_jobs=30)]: Done 102 tasks      | elapsed:  2.2min
[Parallel(n_jobs=30)]: Done 121 tasks      | elapsed:  2.3min
[Parallel(n_jobs=30)]: Done 140 tasks      | elapsed:  2.8min
[Parallel(n_jobs=30)]: Done 161 tasks      | elapsed:  3.3min
[Parallel(n_jobs=30)]: Done 182 tasks      | elapsed:  3.4min
[Parallel(n_jobs=30)]: Done 205 tasks      | elapsed:  3.9min
[Parallel(n_jobs=30)]: Done 228 tasks      | elapsed:  4.4min
[Parallel(n_jobs=30)]: Done 253 tasks      | elapsed:  

~ 9 minutes

In [37]:
import pandas as pd
from IPython.display import HTML, display

# Example: limit rows/columns shown in console
pd.set_option('display.max_rows', 10)
pd.set_option('display.max_columns', 15)
pd.set_option('display.width', 1000)

# Convert DataFrame to HTML and wrap in a scrollable div
html_table = df_results.to_html()
scroll_box = f"""
<div style="max-height:400px; max-width:1000px; overflow-y:auto; overflow-x:auto; border:1px solid #ccc;">
{html_table}
</div>
"""
display(HTML(scroll_box))


,subsample,reg_lambda,reg_alpha,min_child_weight,max_depth,learning_rate,gamma,colsample_bytree,objective,eval_metric,tree_method,seed,nthread,device,verbosity,Unos_C_Index,Std_Dev
16,0.9,0.1,0.1,1,5,0.010,0.1,0.5,survival:cox,cox-nloglik,hist,42,30,cpu,0,0.745007,0.017049
41,0.9,1.0,1.0,10,3,0.010,0.0,0.5,survival:cox,cox-nloglik,hist,42,30,cpu,0,0.744481,0.016503
44,0.7,1.0,0.1,10,4,0.010,0.0,0.6,survival:cox,cox-nloglik,hist,42,30,cpu,0,0.744270,0.015859
32,0.6,1.0,0.0,1,4,0.010,0.1,0.8,survival:cox,cox-nloglik,hist,42,30,cpu,0,0.744245,0.015812
20,0.7,5.0,5.0,10,4,0.005,0.0,0.5,survival:cox,cox-nloglik,hist,42,30,cpu,0,0.744180,0.015198
43,0.8,0.1,0.1,10,3,0.010,0.0,0.8,survival:cox,cox-nloglik,hist,42,30,cpu,0,0.743670,0.015671
12,0.9,1.0,0.1,10,3,0.005,2.0,0.6,survival:cox,cox-nloglik,hist,42,30,cpu,0,0.743611,0.016138
17,0.8,5.0,10.0,1,5,0.010,1.0,0.5,survival:cox,cox-nloglik,hist,42,30,cpu,0,0.743482,0.015742
7,0.8,1.0,10.0,20,8,0.010,2.0,0.5,survival:cox,cox-nloglik,hist,42,30,cpu,0,0.743223,0.016286
19,0.6,0.1,0.1,20,8,0.005,0.0,0.6,survival:cox,cox-nloglik,hist,42,30,cpu,0,0.743158,0.015927


In [38]:
from IPython.display import display, HTML

html_content = """
<div style="font-family: 'Segoe UI', Arial, sans-serif; line-height: 1.6; color: #333; max-width: 850px;">

<h2 style="color: #2c3e50; border-bottom: 2px solid #ecf0f1; padding-bottom: 5px;">🧠 Decoding the XGBoost Grid Search</h2>
<p style="font-size: 15px;">
We tested 50 different configurations to see how XGBoost learns best from the mortality data. 
By looking at what the top models share (and what the bottom models did wrong), the data tells a very clear story about the underlying biology of the patients.
</p>

<table style="width: 100%; border-collapse: collapse; margin-top: 20px;">
    <tr>
        <td style="width: 50%; vertical-align: top; padding-right: 15px;">
            <h3 style="color: #2980b9;">🐢 1. "Slow and Steady" Wins</h3>
            <p style="font-size: 14px;">
                <b>Learning Rate:</b> If you look at the top 15 models, they strictly use low learning rates (<code>0.010</code> or <code>0.005</code>). The models at the very bottom of the table rushed the process with <code>0.100</code>.<br>
                <i>Meaning:</i> The algorithm must take tiny, careful steps to find true mortality risks without tripping over random noise.
            </p>
        </td>
        <td style="width: 50%; vertical-align: top; padding-left: 15px; border-left: 1px solid #eee;">
            <h3 style="color: #2980b9;">🌳 2. Shallow over Deep</h3>
            <p style="font-size: 14px;">
                <b>Max Depth:</b> The best performing trees are very shallow (Depth <code>3</code>, <code>4</code>, or <code>5</code>). Whenever we allowed deep, complex trees (Depth <code>8</code>, seen at the bottom), performance dropped.<br>
                <i>Meaning:</i> Mortality risk is additive and straightforward. Complex, deep decision branches just memorize random patient outliers (overfitting).
            </p>
        </td>
    </tr>
    <tr>
        <td style="width: 50%; vertical-align: top; padding-right: 15px; padding-top: 15px;">
            <h3 style="color: #2980b9;">🛡️ 3. Guardrails for Rare Events</h3>
            <p style="font-size: 14px;">
                <b>Min Child Weight:</b> Many top models favor values like <code>10</code> or <code>20</code>.<br>
                <i>Meaning:</i> This forces the tree to only create a new clinical "rule" if it applies to a solid group of patients, preventing wild guesses based on 1 or 2 isolated deaths.
            </p>
        </td>
        <td style="width: 50%; vertical-align: top; padding-left: 15px; padding-top: 15px; border-left: 1px solid #eee;">
            <h3 style="color: #2980b9;">✂️ 4. Mathematical Penalties</h3>
            <p style="font-size: 14px;">
                <b>Regularization (Alpha/Lambda):</b> The presence of L1 and L2 penalties in the top ranks shows that XGBoost performs better when it is actively punished for adding unnecessary variables.
            </p>
        </td>
    </tr>
</table>

<div style="background-color: #f8f9fa; border-left: 5px solid #27ae60; padding: 15px; margin-top: 25px; border-radius: 0 5px 5px 0;">
    <h4 style="margin-top: 0; color: #27ae60; font-size: 16px;">💡 The Clinical Takeaway</h4>
    <p style="margin-bottom: 0; font-size: 15px;">
        This table proves computationally what we suspected biologically: mortality in this population doesn't have "secret, complex formulas". The risk is driven by strong, direct factors. XGBoost reached high discrimination by acting almost like a traditional linear model—moving slowly, keeping rules simple, and aggressively filtering out noise.
    </p>
</div>

<!-- New compact results box added with minimal change to original layout -->
<div style="background:#fff7e6; border-left:5px solid #f39c12; padding:12px; margin-top:18px; border-radius:4px; max-width:850px;">
  <strong style="color:#d35400;">Recent internal validation update</strong>
  <ul style="margin:8px 0 0 18px; font-size:14px; color:#333;">
    <li>Final tuned model used for validation: <b>Trial 21</b>.</li>
    <li>Optimal boosting rounds (early stopping): <b>708</b>.</li>
    <li>Apparent C-index on full dataset: <b>0.7672</b>.</li>
    <li>Harrell's bootstrap: <b>500</b> resamples run in parallel using <b>30</b> CPU cores.</li>
    <li>Mean optimism (500 boots): <b>0.0184</b>.</li>
    <li>Optimism-corrected C-index (internal validation): <b>0.7488</b>.</li>
    <li>Results saved to: <b>_out/XGB_Death_Bootstrap_Optimism_Results_20260223_1154.csv</b>.</li>
  </ul>
</div>

</div>
"""

display(HTML(html_content))


"🐢 1. ""Slow and Steady"" Wins Learning Rate: If you look at the top 15 models, they strictly use low learning rates (0.010 or 0.005). The models at the very bottom of the table rushed the process with 0.100. Meaning: The algorithm must take tiny, careful steps to find true mortality risks without tripping over random noise.","🌳 2. Shallow over Deep Max Depth: The best performing trees are very shallow (Depth 3, 4, or 5). Whenever we allowed deep, complex trees (Depth 8, seen at the bottom), performance dropped. Meaning: Mortality risk is additive and straightforward. Complex, deep decision branches just memorize random patient outliers (overfitting)."
"🛡️ 3. Guardrails for Rare Events Min Child Weight: Many top models favor values like 10 or 20. Meaning: This forces the tree to only create a new clinical ""rule"" if it applies to a solid group of patients, preventing wild guesses based on 1 or 2 isolated deaths.",✂️ 4. Mathematical Penalties Regularization (Alpha/Lambda): The presence of L1 and L2 penalties in the top ranks shows that XGBoost performs better when it is actively punished for adding unnecessary variables.
